## Run SUMMA model ##

Run the SUMMA base model, with the model runs managed by Snakemake

NOTE that the file manager will need to be updated for this workflow to run!!!
Further testing is needed.


### Write snakemake file ###

This block of code writes the snakemake file

In [2]:
%%writefile ../rules/run_pysumma_ensemble_non_distributed.smk
from pathlib import Path
import sys
import pysumma as ps
import xarray as xr
import shutil
import time

# Add scripts to path
workflow_dir = Path(workflow.basedir).parent
scripts_path = workflow_dir / "scripts"
sys.path.append(str(scripts_path))
import gpep_to_summa_utils as gts_utils

def clean_path(p):
    return Path(str(p).replace(" ", "").replace("\n", "").replace("\t", ""))

# ---------------------------------------
# Load configuration and resolve paths
# ---------------------------------------
config = gts_utils.resolve_paths(config)
CASE_NAME         = config['case_name']
SUMMA_OUTPUT_DIR  = clean_path(config['summa_output_dir'])
FILE_MANAGER      = clean_path(config['file_manager'])
SUMMA_EXE         = clean_path(config['summa_exe'])
SUMMA_FORCING_DIR = clean_path(config['summa_forcing_dir'])

print(f"SUMMA_OUTPUT_DIR: {SUMMA_OUTPUT_DIR}")
print(f"CASE_NAME: {CASE_NAME}")
print(f"FILE_MANAGER: {FILE_MANAGER}")
print(f"SUMMA_EXE: {SUMMA_EXE}")
print(f"SUMMA_FORCING_DIR: {SUMMA_FORCING_DIR}")

# ---------------------------------------
# Ensemble members (subdirectories)
# ---------------------------------------
ens, _ = gts_utils.build_ensemble_list(SUMMA_FORCING_DIR)
ens = [member.replace(" ", "").replace("\n", "").replace("\t", "") for member in ens if member]

print(f"Ensemble members: {ens}")

# ---------------------------------------
# List forcing files (cleaned, Path objects)
# ---------------------------------------
file_paths = gts_utils.list_files_in_subdirectory(SUMMA_FORCING_DIR, filenames_only=True)
file_paths = [clean_path(fp) for fp in file_paths]
print(f"File paths: {[str(fp) for fp in file_paths]}")

# ---------------------------------------
# Create forcing_files_<member>.txt for SUMMA
# ---------------------------------------
forcing_file_dict = {}
for member_id in ens:
    member_id_clean = member_id.replace(" ", "").replace("\n", "").replace("\t", "")
    forcing_list_path = clean_path(SUMMA_FORCING_DIR / f"forcing_files_{member_id_clean}.txt")
    print(f"Forcing list path: {forcing_list_path}")
    forcing_list_path.parent.mkdir(parents=True, exist_ok=True)

    with forcing_list_path.open('w') as fh:
        for fp in file_paths:
            if member_id_clean in fp.parts[0].strip():
                full_path = Path(*[part.strip() for part in fp.parts]).with_suffix(".nc")
                fh.write(full_path.as_posix().strip() + "\n")

    forcing_file_dict[member_id_clean] = forcing_list_path

# ---------------------------------------
# Snakemake Rules
# ---------------------------------------
rule all:
    input:
        expand(str(SUMMA_OUTPUT_DIR / (CASE_NAME + "_{ens_member}_timestep.nc")), ens_member=ens)

rule run_summa_ensemble_simulations:
    input:
        file_manager = FILE_MANAGER,
        forcing_file = lambda wildcards: str(forcing_file_dict[wildcards.ens_member])
    output:
        summa_chunked_output = str(SUMMA_OUTPUT_DIR / (CASE_NAME + "_{ens_member}_timestep.nc"))
    params:
        summa_exe = SUMMA_EXE,
        forcing_path = SUMMA_FORCING_DIR,
        output_path = SUMMA_OUTPUT_DIR,
        run_suffix = lambda wildcards: str(wildcards.ens_member)
    resources:
        runtime=180,
        mem_mb=10000
    run:
        print(f"Running member: {wildcards.ens_member}")
        print(f"Forcing file: {input.forcing_file}")
        forcing_list_name = Path(input.forcing_file).name
        print(f"Forcing list name: {forcing_list_name}")

        sim = ps.Simulation(params.summa_exe, input.file_manager)

        sim_config = {
            'manager': {
                'forcingPath': str(params.forcing_path / params.run_suffix) + '/',
                'outputPath': str(params.output_path) + '/',
                'forcingListFile': str(forcing_list_name)
            }
        }

        sim.apply_config(sim_config)

        forcing_list_output = Path(Path(input.file_manager).parent,'.pysumma',params.run_suffix,forcing_list_name)
        print(f"Copying forcing list from {input.forcing_file} to {forcing_list_output}")
        forcing_list_output.parent.mkdir(parents=True, exist_ok=True)
        time.sleep(2)  # Small delay to help file system sync
        shutil.copy(input.forcing_file,forcing_list_output)
        
        sim.run(run_suffix=str(params.run_suffix),freq_restart='e')
        print(sim.status)


Overwriting ../rules/run_pysumma_ensemble_non_distributed.smk


In [5]:
! snakemake --unlock -s ../rules/run_pysumma_ensemble.smk
! snakemake -s ../rules/run_pysumma_ensemble.smk --cores 10


Unlocking working directory.
Building DAG of jobs...
Using shell: /usr/local/bin/bash
Provided cores: 10
Rules claiming more threads will be scaled down.
Job stats:
job                               count    min threads    max threads
------------------------------  -------  -------------  -------------
run_summa_base_simulations            1              1              1
run_summa_ensemble_simulations        2              1              1
total                                 3              1              1

Select jobs to execute...

[Fri May  9 13:43:53 2025]
rule run_summa_ensemble_simulations:
    input: /anvil/projects/x-ees240082/dcasson/gpep/bow/summa/settings/fileManager.txt, /anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa_ens/002/rf_best_regression_static_dynamic_ensMember_002.nc
    output: /anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa_sim/bow_002_timestep.nc


In [27]:
import pysumma as ps
import xarray as xr
summa_exe = '/Users/dcasson/GitHub/summa/bin/summa.exe'
file_manager = '/Users/dcasson/Data/yukon_esp/summa/settings/fileManagerSWE.txt'
forcing_file = '/Users/dcasson/Data/yukon_esp/summa/forcing/summa/forcing_files_001.txt'

forcing_nc = '/Users/dcasson/Data/gpep/chena/forcing/summa/004/rf_static_boxcox_low_predictors_ensMember_004.nc'
forcing_ds = xr.open_dataset(forcing_nc)
forcing_name = 'test004'
sim = ps.Simulation(summa_exe, file_manager)
sim.assign_forcing_file(forcing_name, forcing_ds)
sim.run()



In [5]:
%%writefile ../rules/run_pysumma_ensemble_distributed.smk
from pathlib import Path
import sys
import pysumma as ps
import shutil
import time

# Add scripts to path
workflow_dir = Path(workflow.basedir).parent
scripts_path = workflow_dir / "scripts"
sys.path.append(str(scripts_path))
import gpep_to_summa_utils as gts_utils
import ss_utils

def clean_path(p):
    return Path(str(p).replace(" ", "").replace("\n", "").replace("\t", ""))

# --- Load configuration ---
config = gts_utils.resolve_paths(config)
CASE_NAME         = config['case_name']
SUMMA_OUTPUT_DIR  = clean_path(config['summa_output_dir'])
FILE_MANAGER      = clean_path(config['file_manager'])
SUMMA_EXE         = clean_path(config['summa_exe'])
SUMMA_FORCING_DIR = clean_path(config['summa_forcing_dir'])
ATTRIBUTES_NC     = clean_path(config['attribute_nc'])
GRU_CHUNK_SIZE    = int(config.get('gru_chunk_size', 10))

# --- Ensemble members ---
ens, _ = gts_utils.build_ensemble_list(SUMMA_FORCING_DIR)
ens = [member.strip() for member in ens if member]

# --- GRU chunks ---
num_grus = ss_utils.calc_num_grus(ATTRIBUTES_NC)
gru_chunks = ss_utils.generate_gru_start_and_count(num_grus, chunk_size=GRU_CHUNK_SIZE)

# --- Forcing file lists per ensemble member ---
file_paths = gts_utils.list_files_in_subdirectory(SUMMA_FORCING_DIR, filenames_only=True)
file_paths = [clean_path(fp) for fp in file_paths]

forcing_file_dict = {}
for member_id in ens:
    member_id_clean = member_id.strip()
    forcing_list_path = clean_path(SUMMA_FORCING_DIR / f"forcing_files_{member_id_clean}.txt")
    forcing_list_path.parent.mkdir(parents=True, exist_ok=True)
    with forcing_list_path.open('w') as fh:
        for fp in file_paths:
            if member_id_clean in fp.parts[0]:
                full_path = Path(*[part.strip() for part in fp.parts]).with_suffix(".nc")
                fh.write(full_path.as_posix() + "\n")
    forcing_file_dict[member_id_clean] = forcing_list_path

# --- Rules ---

rule all:
    input:
        expand(
            str(SUMMA_OUTPUT_DIR / f"{CASE_NAME}_{{ens_member}}_timestep.nc"),
            ens_member=ens,
        )

# Run SUMMA in GRU chunks for each ensemble member
rule run_summa_ensemble_gru_chunk:
    input:
        file_manager = FILE_MANAGER,
        forcing_file = lambda wildcards: str(forcing_file_dict[wildcards.ens_member])
    output:
        chunked_output = temp(str(SUMMA_OUTPUT_DIR / f"{CASE_NAME}_{{ens_member}}_{{gru_chunk}}_timestep.nc"))
    params:
        summa_exe = SUMMA_EXE,
        forcing_path = SUMMA_FORCING_DIR,
        output_path = SUMMA_OUTPUT_DIR,
        run_suffix = lambda wildcards: wildcards.ens_member,
        gru_start = lambda wildcards: int(wildcards.gru_chunk.split('_')[0]),
        gru_count = lambda wildcards: int(wildcards.gru_chunk.split('_')[1])
    resources:
        runtime=180,
        mem_mb=10000
    run:
        print(f"Running member: {wildcards.ens_member}, GRU chunk: {wildcards.gru_chunk}")
        sim = ps.Simulation(params.summa_exe, input.file_manager)
        sim_config = {
            'manager': {
                'forcingPath': str(params.forcing_path / params.run_suffix) + '/',
                'outputPath': str(params.output_path) + '/',
                'forcingListFile': Path(input.forcing_file).name
            }
        }
        sim.apply_config(sim_config)

        # Copy forcing list for SUMMA
        forcing_list_name = Path(input.forcing_file).name
        forcing_list_output = Path(Path(input.file_manager).parent, '.pysumma', params.run_suffix, forcing_list_name)
        forcing_list_output.parent.mkdir(parents=True, exist_ok=True)
        time.sleep(2)
        if not forcing_list_output.exists():
            shutil.copy(input.forcing_file, forcing_list_output)
            print(f"Copied to {forcing_list_output}")
        else:
            print(f"{forcing_list_output} already exists, skipping copy.")

        # Run SUMMA for the GRU chunk
        sim.run(
            run_suffix=str(params.run_suffix),
            startGRU=params.gru_start,
            countGRU=params.gru_count,
            freq_restart='e'
        )
        print(sim.status)

# Merge SUMMA chunk outputs for each ensemble member
rule merge_summa_output_files:
    input:
        chunked_outputs = expand(
            str(SUMMA_OUTPUT_DIR / f"{CASE_NAME}_{{ens_member}}_{{gru_chunk}}_timestep.nc"),
            gru_chunk=gru_chunks
        )
    output:
        merged_output = str(SUMMA_OUTPUT_DIR / f"{CASE_NAME}_{{ens_member}}_timestep.nc")
    run:
        print(f"Merging SUMMA outputs for {wildcards.ens_member} into {output.merged_output}.")
        ss_utils.merge_netcdf_output(input.chunked_outputs, output.merged_output)

Overwriting ../rules/run_pysumma_ensemble_distributed.smk


In [15]:
%%writefile ../rules/run_fsca_simulations.smk
"""

Snakemake file to calculate fSCA based on SUMMA model simulations.

"""

from pathlib import Path
import sys
# Import local packages
sys.path.append(str(Path('../').resolve()))
sys.path.append(str(Path('../../').resolve()))
sys.path.append(str(Path('../../snow_dist').resolve()))
from scripts import ss_utils
from snow_dist import summa_simulation

# Resolve all file paths and directories in the config file
config['summa_output_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['working_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['summa_interim_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/interim'
config['fsca_output_dir'] =  '/Users/drc858/Data/gpep/RF_ens/summa/output/fsca'
config_file_path = '/Users/drc858/GitHub/snow_dist/settings/config_summa_model_tuolumne.yaml'

#Read first raw forcing file for easymore remapping
summa_result_filepaths = list(Path(config['summa_output_dir']).glob('*.nc'))
summa_filenames = [Path(filepath).name for filepath in summa_result_filepaths]
print(summa_filenames)

fsca_simulation = summa_simulation.SummaSimulation(config_file_path)

# Prepare summa model results for fsca
# This includes summing the input and output snowpack variables
rule run_fsca_model:
    input:
        expand(Path(config['summa_interim_dir'], "{filenames}"))

rule prepare_fsca_model_input:
    input:
        summa_result = Path(config['summa_output_dir'],"{filenames}")
    output:
        fsca_input = Path(config['summa_interim_dir'],"{filenames}")
    run:
        summa_ds = xr.open_dataset(input.input_summa_result)
        fsca_ds = summa_fsca_model.prepare_summa_fsca_input(summa_ds)
        fsca_ds.to_netcdf(output.fsca_input)

# Run fsca settings
rule run_fsca_simulations:
    input:
        prepped_files = expand(Path(config['summa_interim_dir'], "{filenames}"), filenames=summa_filenames)
    output:
        fsca_result = Path(config['fsca_output_dir'], "{filenames}")
    run:
        sim = summa_simulation.SummaSimulation(config)
        summa_fsca_model.run_summa_fsca_model(input.prepped_files,output.fsca_result)



Overwriting ../rules/run_fsca_simulations.smk


In [16]:
#-s ../rules/run_pysumma_snakemake.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml 
! snakemake -s ../rules/run_fsca_simulations.smk --cores 8 --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml

ImportError in file /Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk, line 13:
cannot import name 'ss_utils' from 'scripts' (unknown location)
  File "/Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk", line 13, in <module>


In [19]:
import yaml
from pathlib import Path
import shutil
import sys
import time
import pysumma as ps

# --- Add scripts to path if needed ---
scripts_path = Path('../scripts')  # Update if needed
sys.path.append(str(scripts_path))
import gpep_to_summa_utils as gts_utils
import ss_utils

def clean_path(p):
    return Path(str(p).replace(" ", "").replace("\n", "").replace("\t", ""))

def parse_gru_chunk(chunk_str):
    chunk_str = chunk_str.strip('G')
    start_str, end_str = chunk_str.split('-')
    gru_start = int(start_str)
    gru_end = int(end_str)
    gru_count = gru_end - gru_start + 1
    return gru_start, gru_count

# --- Load configuration from YAML ---
def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

# --- Test case: Point to your config.yaml file ---
config_path = "/home/x-dcasson/GitRepos/gpep_to_summa_snakemake/workflow/config/gpep_to_summa_bow.yaml"  # UPDATE THIS
config = load_config(config_path)
config = gts_utils.resolve_paths(config)  # If present

CASE_NAME         = config['case_name']
SUMMA_OUTPUT_DIR  = clean_path(config['summa_output_dir'])
FILE_MANAGER      = clean_path(config['file_manager'])
SUMMA_EXE         = clean_path(config['summa_exe'])
SUMMA_FORCING_DIR = clean_path(config['summa_forcing_dir'])
ATTRIBUTES_NC     = clean_path(config['attribute_nc'])
GRU_CHUNK_SIZE    = int(config.get('gru_chunk_size', 50))

# --- Ensemble members ---
ens, _ = gts_utils.build_ensemble_list(SUMMA_FORCING_DIR)
ens = [member.strip() for member in ens if member]
print("Ensemble members:", ens)

# --- GRU chunks ---
num_grus = ss_utils.calc_num_grus(ATTRIBUTES_NC)
gru_chunks = ss_utils.generate_gru_start_and_count(num_grus, chunk_size=GRU_CHUNK_SIZE)
print("GRU chunks:", gru_chunks)

# --- Forcing file lists per ensemble member ---
file_paths = gts_utils.list_files_in_subdirectory(SUMMA_FORCING_DIR, filenames_only=True)
file_paths = [clean_path(fp) for fp in file_paths]

forcing_file_dict = {}
for member_id in ens:
    member_id_clean = member_id.strip()
    forcing_list_path = clean_path(SUMMA_FORCING_DIR / f"forcing_files_{member_id_clean}.txt")
    forcing_list_path.parent.mkdir(parents=True, exist_ok=True)
    with forcing_list_path.open('w') as fh:
        for fp in file_paths:
            if member_id_clean in fp.parts[0]:
                full_path = Path(*[part.strip() for part in fp.parts]).with_suffix(".nc")
                fh.write(full_path.as_posix() + "\n")
    forcing_file_dict[member_id_clean] = forcing_list_path

print("Forcing file dict:", forcing_file_dict)

# === ACTUALLY RUN SUMMA FOR ONE CHUNK ===

test_ens_member = ens[0]        # Or pick desired member
test_gru_chunk = gru_chunks[0]  # Or pick desired chunk
gru_start, gru_count = parse_gru_chunk(test_gru_chunk)

print(f"\nRunning SUMMA for member {test_ens_member}, chunk {test_gru_chunk} (start={gru_start}, count={gru_count})")

forcing_file = forcing_file_dict[test_ens_member]
forcing_list_name = Path(forcing_file).name

# Unique per-job forcing list copy to avoid concurrency issues
forcing_list_output = Path(
    Path(FILE_MANAGER).parent,
    '.pysumma',
    f"{test_ens_member}_{test_gru_chunk}",
    forcing_list_name
)
forcing_list_output.parent.mkdir(parents=True, exist_ok=True)
time.sleep(1)
if not forcing_list_output.exists():
    shutil.copy(forcing_file, forcing_list_output)
    print(f"Copied to {forcing_list_output}")
else:
    print(f"{forcing_list_output} already exists, skipping copy.")

# SUMMA output path for this test
chunked_output = SUMMA_OUTPUT_DIR / f"{CASE_NAME}_{test_ens_member}_{test_gru_chunk}_timestep.nc"
chunked_output.parent.mkdir(parents=True, exist_ok=True)

# Set up simulation
sim = ps.Simulation(str(SUMMA_EXE), str(FILE_MANAGER))

sim_config = {
    'manager': {
        'forcingPath': str(SUMMA_FORCING_DIR / test_ens_member) + '/',
        'outputPath': str(SUMMA_OUTPUT_DIR) + '/',
        'forcingListFile': forcing_list_name
    }
}
sim.apply_config(sim_config)

#check if attribute file exists

# Check if attribute file exists
#attribute_file_path = Path(ATTRIBUTES_NC)
attribute_file_path = Path(Path(FILE_MANAGER).parent,'attributes.nc')
if attribute_file_path.exists():
    print(f"Attribute file found at: {attribute_file_path}")
    # Actually run SUMMA for this GRU chunk!
    sim.run(
        run_suffix=str(test_ens_member),
        startGRU=gru_start,
        countGRU=gru_count,
        write_config=False,
        freq_restart='e'
    )
else:
    print(f"Warning: Attribute file not found at: {attribute_file_path}")
    print("Will set write_config=True to generate default configuration")
    # Actually run SUMMA for this GRU chunk!
sim.run(
    run_suffix=str(test_ens_member),
    startGRU=gru_start,
    countGRU=gru_count,
    write_config=True,
    freq_restart='e'
)


print("SUMMA run status:", sim.status)

Ensemble members: ['001', '002']
GRU chunks: ['G0001-0050', 'G0051-0100', 'G0101-0150', 'G0151-0200', 'G0201-0250', 'G0251-0300', 'G0301-0350', 'G0351-0400', 'G0401-0450', 'G0451-0500', 'G0501-0550', 'G0551-0600', 'G0601-0650', 'G0651-0700', 'G0701-0750', 'G0751-0800', 'G0801-0850', 'G0851-0900', 'G0901-0950', 'G0951-1000', 'G1001-1050', 'G1051-1100', 'G1101-1150', 'G1151-1181']
Forcing file dict: {'001': PosixPath('/anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa/forcing/forcing_files_001.txt'), '002': PosixPath('/anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa/forcing/forcing_files_002.txt')}

Running SUMMA for member 001, chunk G0001-0050 (start=1, count=50)
/anvil/projects/x-ees240082/dcasson/gpep/bow/summa/settings/.pysumma/001_G0001-0050/forcing_files_001.txt already exists, skipping copy.
Attribute file found at: /anvil/projects/x-ees240082/dcasson/gpep/bow/summa/setti